# MedSAM2 inference on RexGroundingCT — lung nodule (2d) / GGO (2c)

This notebook runs **MedSAM2** on the 163-case RexGroundingCT test subset whose findings
are exclusively lung-nodule (`2d`) or ground-glass opacity (`2c`) —
`data/rexgrounding-ct/dataset_2_last.json`, `test` split (the same subset
`models/lc-ksvd` treats as canonical).

**Important caveat:** MedSAM2 is a point/box-*prompted* 3D video segmenter, not a
detector — it has no notion of "nodule" vs "opacity" on its own. For every GT finding
we take the largest-area slice of its ground-truth mask, build a 2D box prompt from it,
and propagate forward/backward through the volume. So this measures MedSAM2's
**segmentation quality given a correct localization**, not its ability to find lesions
unassisted.

Before running: download checkpoints with `bash download.sh` from this folder
(`models/med-sam2/checkpoints/MedSAM2_latest.pt` is used below).

In [ ]:
# Setup — run this notebook from models/med-sam2/notebooks (working dir is fixed below)
import os, sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
MEDSAM2_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
os.chdir(MEDSAM2_DIR)  # sam2's hydra config module resolves configs relative to this cwd
sys.path.insert(0, str(MEDSAM2_DIR))
print("Working dir:", Path.cwd())

In [ ]:
import json

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import SimpleITK as sitk
import torch
from PIL import Image
from skimage import measure

from sam2.build_sam import build_sam2_video_predictor_npz
from rexgroundingct_dataset import load_cases, RexCase

torch.set_float32_matmul_precision("high")
torch.manual_seed(2024)
if torch.cuda.is_available():
    torch.cuda.manual_seed(2024)
np.random.seed(2024)

## Configure paths

- `checkpoint`: MedSAM2 weights (`MedSAM2_latest.pt` recommended; `MedSAM2_CTLesion.pt` is the
  CT-lesion-finetuned alternative)
- `cfg`: model config, relative to this folder
- `output_dir`: where `reports.json` + `masks/*.npz` are written (consumed by
  `eval/adapters/medsam2_adapter.py` via `eval.runners.evaluate_localization --model medsam2`)

In [ ]:
checkpoint = "checkpoints/MedSAM2_latest.pt"
cfg = "configs/sam2.1_hiera_t512.yaml"
output_dir = Path("../outputs/medsam2_rexgroundingct")  # -> code/outputs/medsam2_rexgroundingct
(output_dir / "masks").mkdir(parents=True, exist_ok=True)

assert Path(checkpoint).exists(), f"Checkpoint not found: {checkpoint}. Run bash download.sh first."

IMG_MEAN = (0.485, 0.456, 0.406)
IMG_STD = (0.229, 0.224, 0.225)
IMSIZE = 512

In [ ]:
predictor = build_sam2_video_predictor_npz(cfg, checkpoint)

## Load the 163-case test subset

In [ ]:
cases = load_cases()
print(f"Loaded {len(cases)} cases")
cases[0]

## Helper functions

Same preprocessing/prompting conventions as the shipped `medsam2_infer_3D_CT.py` /
`examples/infer_CT_LUNA25.py` scripts: HU windowing to `[0, 255]`, resize in-plane to
512x512 RGB, ImageNet-style normalization, box prompt on the key slice, then
forward + backward propagation via `propagate_in_video`.

GT masks are stored as a unified 4D volume `(F, X, Y, Z)` (nibabel axis order); CT
volumes are read with SimpleITK, which returns `(Z, Y, X)`. `load_gt_mask_zyx` transposes
the mask to `(F, Z, Y, X)` so mask and image slices line up during prompting; the final
saved prediction is transposed back to `(X, Y, Z)` to match the GT convention the
evaluator expects.

In [ ]:
def window_ct(volume, level=-750.0, width=1500.0):
    lower, upper = level - width / 2.0, level + width / 2.0
    clipped = np.clip(volume.astype(np.float32), lower, upper)
    return (clipped - lower) / (upper - lower) * 255.0


def resize_grayscale_to_rgb(array, image_size):
    d, h, w = array.shape
    out = np.zeros((d, 3, image_size, image_size), dtype=np.float32)
    for i in range(d):
        img = Image.fromarray(array[i].astype(np.uint8)).convert("RGB")
        img = img.resize((image_size, image_size))
        out[i] = np.array(img).transpose(2, 0, 1)
    return out


def load_gt_mask_zyx(mask_path):
    """Load the unified (F, X, Y, Z) mask and return it as (F, Z, Y, X)."""
    mask = np.asarray(nib.load(str(mask_path)).dataobj, dtype=np.uint8)
    return np.transpose(mask, (0, 3, 2, 1))


def finding_box(mask_f_zyx, margin=3):
    """Largest-area slice for this finding -> (key_slice, box=[x0, y0, x1, y1])."""
    areas = mask_f_zyx.reshape(mask_f_zyx.shape[0], -1).sum(axis=1)
    key_slice = int(np.argmax(areas))
    y_idx, x_idx = np.where(mask_f_zyx[key_slice] > 0)
    h, w = mask_f_zyx.shape[1:]
    x_min = max(0, int(x_idx.min()) - margin)
    x_max = min(w - 1, int(x_idx.max()) + margin)
    y_min = max(0, int(y_idx.min()) - margin)
    y_max = min(h - 1, int(y_idx.max()) + margin)
    return key_slice, np.array([x_min, y_min, x_max, y_max])


def largest_cc(mask):
    if mask.sum() == 0:
        return mask
    labels = measure.label(mask)
    counts = np.bincount(labels.flat)
    counts[0] = 0
    return (labels == np.argmax(counts)).astype(np.uint8)


def show_mask(mask, ax, color=(251/255, 252/255, 30/255), alpha=0.5):
    h, w = mask.shape[-2:]
    overlay = mask.reshape(h, w, 1) * np.array([*color, alpha]).reshape(1, 1, -1)
    ax.imshow(overlay)


def show_box(box, ax, edgecolor="blue"):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor=edgecolor, facecolor=(0, 0, 0, 0), lw=2))

In [ ]:
@torch.inference_mode()
def segment_finding(img_resized, video_height, video_width, key_slice, box, num_frames):
    segs = np.zeros((num_frames, video_height, video_width), dtype=np.uint8)
    autocast_kwargs = dict(dtype=torch.bfloat16, enabled=torch.cuda.is_available())

    with torch.autocast("cuda", **autocast_kwargs):
        inference_state = predictor.init_state(img_resized, video_height, video_width)
        predictor.add_new_points_or_box(
            inference_state=inference_state, frame_idx=key_slice, obj_id=1, box=box,
        )
        for out_frame_idx, _, out_mask_logits in predictor.propagate_in_video(
            inference_state, start_frame_idx=key_slice, reverse=False
        ):
            segs[out_frame_idx, (out_mask_logits[0] > 0.0).cpu().numpy()[0]] = 1
        predictor.reset_state(inference_state)

        inference_state = predictor.init_state(img_resized, video_height, video_width)
        predictor.add_new_points_or_box(
            inference_state=inference_state, frame_idx=key_slice, obj_id=1, box=box,
        )
        for out_frame_idx, _, out_mask_logits in predictor.propagate_in_video(
            inference_state, start_frame_idx=key_slice, reverse=True
        ):
            segs[out_frame_idx, (out_mask_logits[0] > 0.0).cpu().numpy()[0]] = 1
        predictor.reset_state(inference_state)

    return segs


def load_case_tensors(case: RexCase):
    sitk_img = sitk.ReadImage(str(case.volume_path))
    img_zyx = sitk.GetArrayFromImage(sitk_img)  # (Z, Y, X)
    img_pre = window_ct(img_zyx)
    assert img_pre.max() < 256

    d, h, w = img_pre.shape
    if h != IMSIZE or w != IMSIZE:
        img_resized = resize_grayscale_to_rgb(img_pre, IMSIZE)
    else:
        img_resized = img_pre[:, None].repeat(3, axis=1)
    img_resized = torch.from_numpy(img_resized / 255.0).float()
    if torch.cuda.is_available():
        img_resized = img_resized.cuda()
    mean = torch.tensor(IMG_MEAN, dtype=torch.float32)[:, None, None].to(img_resized.device)
    std = torch.tensor(IMG_STD, dtype=torch.float32)[:, None, None].to(img_resized.device)
    img_resized = (img_resized - mean) / std

    return sitk_img, img_zyx, img_pre, img_resized, d, h, w


def run_case(case: RexCase, keep_largest_cc=True):
    sitk_img, img_zyx, img_pre, img_resized, d, h, w = load_case_tensors(case)
    gt_mask = load_gt_mask_zyx(case.mask_path)  # (F, Z, Y, X)

    class_volumes = {}
    predictions = {}
    prompts = []  # for visualization: (finding, key_slice, box)

    for finding in case.findings:
        if finding.index >= gt_mask.shape[0]:
            continue
        mask_f = gt_mask[finding.index]
        if mask_f.sum() == 0:
            continue

        key_slice, box = finding_box(mask_f)
        prompts.append((finding, key_slice, box))
        segs = segment_finding(img_resized, h, w, key_slice, box, d)
        if keep_largest_cc:
            segs = largest_cc(segs)

        class_name = finding.class_name
        if class_name not in class_volumes:
            class_volumes[class_name] = np.zeros_like(segs)
        class_volumes[class_name] = np.logical_or(class_volumes[class_name], segs).astype(np.uint8)

    for class_name, vol_zyx in class_volumes.items():
        vol_xyz = np.transpose(vol_zyx, (2, 1, 0))  # match nib (X, Y, Z) GT convention
        predictions[class_name] = {
            "existence_score": 1.0 if vol_xyz.sum() > 0 else 0.0,
            "mask_voxels": float(vol_xyz.sum()),
        }
        class_volumes[class_name] = vol_xyz

    spacing_xyz = sitk_img.GetSpacing()
    return predictions, class_volumes, spacing_xyz, img_pre, gt_mask, prompts

## Demo: run on a single case and visualize the box prompt + predicted mask

In [ ]:
demo_case = cases[0]
print(demo_case.case_id, [(f.index, f.category, f.text) for f in demo_case.findings])

predictions, class_volumes, spacing_xyz, img_pre, gt_mask, prompts = run_case(demo_case)
predictions

In [ ]:
finding, key_slice, box = prompts[0]
pred_vol_xyz = class_volumes[finding.class_name]
pred_slice = pred_vol_xyz[:, :, key_slice].T  # back to (Y, X) for display
gt_slice = gt_mask[finding.index, key_slice]  # (Y, X)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, mask, title in zip(axes, [gt_slice, pred_slice], ["GT + box prompt", "MedSAM2 prediction"]):
    ax.imshow(img_pre[key_slice], cmap="gray")
    show_mask(mask, ax, color=(1, 0, 0) if title.startswith("GT") else (0, 1, 0))
    show_box(box, ax)
    ax.set_title(title)
    ax.axis("off")
plt.suptitle(f"{demo_case.case_id} — finding {finding.index} ({finding.class_name})")
plt.tight_layout()
plt.show()

## Full batch inference over the 163-case subset

Writes `reports.json` + `masks/<case_id>.npz` to `output_dir`, matching the contract
`eval/adapters/medsam2_adapter.py` expects. Set `limit` to a small number first to
sanity-check timing before committing to the full run (a case with N findings runs
MedSAM2 propagation N times, so total time scales with total finding count, not case
count).

In [ ]:
limit = 0  # 0 = run all 163 cases; set e.g. 5 for a quick smoke test
keep_largest_cc = True

run_cases = cases[:limit] if limit > 0 else cases
reports = []

for i, case in enumerate(run_cases, 1):
    print(f"[{i}/{len(run_cases)}] {case.case_id}", flush=True)
    predictions, class_volumes, spacing_xyz, *_ = run_case(case, keep_largest_cc=keep_largest_cc)

    mask_file = output_dir / "masks" / f"{case.case_id}.npz"
    np.savez_compressed(mask_file, **class_volumes)

    reports.append({
        "case_id": case.case_id,
        "volume_name": case.volume_name,
        "protocol": case.protocol,
        "predictions": predictions,
        "artifacts": {
            "mask_file": str(mask_file.relative_to(output_dir)),
            "volume_metadata": {"spacing_xyz": list(spacing_xyz)},
        },
    })

report_path = output_dir / "reports.json"
report_path.write_text(json.dumps(reports, indent=2), encoding="utf-8")
print(f"Wrote report: {report_path}")

## Evaluate

From the repo root (`/home/chest_ct/code`):

```bash
python -m eval.runners.evaluate_localization \\
    --model medsam2 \\
    --predictions-dir outputs/medsam2_rexgroundingct \\
    --gt-mask-root data/segmentations/segmentations \\
    --metadata-json data/rexgrounding-ct/dataset_2_last.json \\
    --output-dir outputs/eval/localization
```

This reuses the same runner as `biomed_parse` / `merlin`, so results land in the same
`by_class` / `by_morphology` / `threshold_sweep` comparison format.